In [ ]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                self.p.v_pd_hf_tweezer_squeeze_power = 1.97
                                 
                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 500.e-6
                
                self.p.amp_imaging = {img_amp}
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 1.2 * np.pi

                self.p.t_tweezer_hold = 15.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 15

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_midpoint)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(ramp_down_painting=True,squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)

                self.raman.pulse(t=self.p.t_continuous_rabi)
                
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [2]:
eBuilder = ExptBuilder()

In [ ]:
img_amp = np.linspace(.2,2.5,50)

freq_raman = np.array([1.19501126e+08, 1.19509811e+08, 1.19518496e+08, 1.19527180e+08,
       1.19535865e+08, 1.19544549e+08, 1.19553234e+08, 1.19561918e+08,
       1.19570603e+08, 1.19579288e+08, 1.19587972e+08, 1.19596657e+08,
       1.19605341e+08, 1.19614026e+08, 1.19622710e+08, 1.19631395e+08,
       1.19640079e+08, 1.19648764e+08, 1.19657449e+08, 1.19666133e+08,
       1.19674818e+08, 1.19683502e+08, 1.19692187e+08, 1.19700871e+08,
       1.19709556e+08, 1.19718241e+08, 1.19726925e+08, 1.19735610e+08,
       1.19744294e+08, 1.19752979e+08, 1.19761663e+08, 1.19770348e+08,
       1.19779032e+08, 1.19787717e+08, 1.19796402e+08, 1.19805086e+08,
       1.19813771e+08, 1.19822455e+08, 1.19831140e+08, 1.19839824e+08,
       1.19848509e+08, 1.19857194e+08, 1.19865878e+08, 1.19874563e+08,
       1.19883247e+08, 1.19891932e+08, 1.19900616e+08, 1.19909301e+08,
       1.19917985e+08, 1.19926670e+08])

# img_amp = np.linspace(.5,1.5,50)

# freq_raman = np.array([1.19556632e+08, 1.19560408e+08, 1.19564184e+08, 1.19567960e+08,
#        1.19571736e+08, 1.19575512e+08, 1.19579288e+08, 1.19583063e+08,
#        1.19586839e+08, 1.19590615e+08, 1.19594391e+08, 1.19598167e+08,
#        1.19601943e+08, 1.19605719e+08, 1.19609495e+08, 1.19613271e+08,
#        1.19617047e+08, 1.19620822e+08, 1.19624598e+08, 1.19628374e+08,
#        1.19632150e+08, 1.19635926e+08, 1.19639702e+08, 1.19643478e+08,
#        1.19647254e+08, 1.19651030e+08, 1.19654805e+08, 1.19658581e+08,
#        1.19662357e+08, 1.19666133e+08, 1.19669909e+08, 1.19673685e+08,
#        1.19677461e+08, 1.19681237e+08, 1.19685013e+08, 1.19688789e+08,
#        1.19692564e+08, 1.19696340e+08, 1.19700116e+08, 1.19703892e+08,
#        1.19707668e+08, 1.19711444e+08, 1.19715220e+08, 1.19718996e+08,
#        1.19722772e+08, 1.19726547e+08, 1.19730323e+08, 1.19734099e+08,
#        1.19737875e+08, 1.19741651e+08])


for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.5 119556632.0
0  15 values of dummy. 15 total shots. 45 total images expected.
Run ID: 71381
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 1.2, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 1.2 pi, x-center = 994, y-center = 820

 Run ID: 71381

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 1.2, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 1.2 pi, x-center = 994, y-center = 820

laser ry_405 unlocked:
target = 741.092263, meas = 741.091334, diff = -929.0
laser ry_980 unlocked:
target = 307.420122, meas = 307.420339, diff = 217.0
shot 1/15 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 1.2, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 1.2 pi, x-center = 994, y-center = 820

laser ry_405 unlocked:
target = 741.092263, mea

In [4]:
from kexp import EthernetRelay
from waxx.util.guis.als.als_gui_client import ALSGuiClient
from waxx.util.guis.precilaser.precilaser_gui_client import PrecilaserGuiClient

relay = EthernetRelay()
relay.source_off()

als = ALSGuiClient()
ok = als.run_shutdown_sequence()

precilaser = PrecilaserGuiClient()
ok = precilaser.run_shutdown_sequence()